<a href="https://colab.research.google.com/github/kaacd/KomputasiNumerik/blob/main/Muhammad_Arkananta_Firdaus_202511420006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Tambahkan nilai EPSILON untuk batas toleransi nol
EPSILON = 1e-10

import numpy as np: NumPy adalah pustaka andalan di Python untuk matematika. Bayangkan ini seperti kalkulator super yang bisa menghitung matriks (tabel angka) dengan sangat cepat.

EPSILON = 1e-12: Ini adalah angka yang sangat kecil ($0.000000000001$). Di dunia komputer, hasil hitungan desimal kadang tidak persis 0 (misalnya 0.0000000000001). Jadi, kita menggunakan EPSILON sebagai batas. Jika ada angka yang lebih kecil dari EPSILON, komputer akan menganggapnya sama dengan nol.

In [2]:
def gauss_eliminasi(A, b, tampil_langkah=True):
    """Menyelesaikan A.x = b dengan eliminasi Gauss."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    # Membentuk matriks augmented [A | b]
    M = np.hstack((A, b.reshape(-1, 1)))
    np.set_printoptions(precision=4, suppress=True, linewidth=120)

    if tampil_langkah:
        print("Matriks augmented awal:")
        print(M, "\n")

    # ---------- Tahap 1: Eliminasi maju ----------
    for k in range(n - 1):
        # Partial pivoting: cari baris dengan nilai mutlak terbesar
        p = k + np.argmax(np.abs(M[k:, k]))

        # Cek jika pivot bernilai 0 (tidak ada solusi tunggal)
        if abs(M[p, k]) < EPSILON:
            print("SPL tidak memiliki solusi tunggal (pivot = 0).")
            return None

        # Tukar baris jika nilai terbesar tidak ada di atas
        if p != k:
            M[[k, p]] = M[[p, k]]
            if tampil_langkah:
                print(f"Tukar baris R{k+1} <-> R{p+1}")

        # Proses membuat nol angka di bawah pivot
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            M[i, k:] = M[i, k:] - m * M[k, k:]
            if tampil_langkah:
                print(f"R{i+1} <- R{i+1} - ({m:.4f}) * R{k+1}")

        if tampil_langkah:
            print(f"\nSetelah eliminasi kolom {k+1}:")
            print(M, "\n")

    if abs(M[n - 1, n - 1]) < EPSILON:
        print("SPL tidak memiliki solusi tunggal (pivot = 0).")
        return None

    # ---------- Tahap 2: Substitusi mundur ----------
    x = np.zeros(n)
    x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]

    for i in range(n - 2, -1, -1):
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - jumlah) / M[i, i]

    return x

Ini adalah mesin utama program kita. Fungsi gauss_eliminasi bekerja dalam beberapa langkah:

Matriks Augmented: Menggabungkan persamaan di sisi kiri ($A$) dan sisi kanan ($b$) menjadi satu tabel besar (M) agar mudah dihitung sekaligus.

Eliminasi Maju: Proses ini membuat bentuk "segitiga bawah" matriks menjadi angka nol.

Partial Pivoting: Sebelum baris dikurangi, program akan mencari angka terbesar di kolom tersebut dan menukarnya ke posisi atas (pivot). Ini mencegah error pada komputer (menghindari pembagian dengan angka yang mendekati nol).

Substitusi Mundur: Setelah matriks berbentuk seperti tangga, program mencari nilai variabel dari yang paling bawah (misalnya $z$), lalu naik ke atas untuk mencari $y$, dan terakhir mencari $x$.

Membuat Nol (Eliminasi): Ini adalah inti dari metode Gauss. Kita ingin membuat angka di bawah pivot menjadi  0 .

m adalah faktor pengali (angka yang mau dinolkan dibagi dengan angka pivot).

Baris saat ini (M[i, k:]) dikurangi dengan hasil kali m terhadap baris pivot (M[k, k:]). (Catatan: Di kode asli Anda, tanda minus - hilang di bagian ini).

Mencari Variabel Terakhir: Kita membuat array x berisi angka nol untuk menyimpan jawaban. Nilai variabel paling terakhir (x[n-1]) dicari terlebih dahulu dengan cara membagi kolom hasil (paling kanan) dengan koefisiennya.

Substitusi Naik: Setelah nilai variabel paling bawah ketemu, kita "naik" ke baris atasnya.

np.dot berfungsi mengalikan koefisien dengan nilai x yang sudah kita temukan sebelumnya, lalu menjumlahkannya.

Variabel saat ini (x[i]) dihitung dengan rumus: (Nilai paling kanan dikurangi total nilai variabel yang sudah diketahui) dibagi dengan koefisien variabel saat ini. (Catatan: Di kode asli Anda, tanda minus - juga hilang di bagian ini).

In [3]:
def baca_input():
    """Membaca SPL dari input pengguna."""
    n = int(input("Masukkan jumlah persamaan/variabel (n): "))
    print(f"\nMasukkan koefisien matriks A ({n} baris, pisahkan dengan spasi):")
    A = []
    for i in range(n):
        baris = list(map(float, input(f"  Baris {i+1}: ").split()))
        if len(baris) != n:
            raise ValueError(f"Baris harus berisi {n} angka.")
        A.append(baris)
    print(f"\nMasukkan vektor konstanta b ({n} angka, pisahkan dengan spasi):")
    b = list(map(float, input("  b: ").split()))
    if len(b) != n:
        raise ValueError(f"Vektor b harus berisi {n} angka.")
    return A, b

baca_input(): Fungsi ini bertugas membaca angka yang kamu ketik di keyboard jika kamu memilih untuk memasukkan soal buatanmu sendiri.

In [4]:
def jalankan_aplikasi():
    """Fungsi pembungkus untuk menjalankan menu."""
    print("=== SPL - Metode Eliminasi Gauss ===")
    print("1. Gunakan contoh soal")
    print("2. Input sendiri")
    pilihan = input("Pilih (1/2): ").strip()

    if pilihan == "2":
        A, b = baca_input()
    else:
        # Contoh persamaan bawaan
        A = [[2, 1, -1],
             [-3, -1, 2],
             [-2, 1, 2]]
        b = [8, -11, -3]
        print("\nContoh soal yang digunakan:")
        for i in range(len(b)):
            print("  ", A[i], "| ", b[i])
        print()

    # Memanggil fungsi eliminasi yang ada di Sel 2
    x = gauss_eliminasi(A, b)

    if x is not None:
        print("=== HASIL ===")
        for i, nilai in enumerate(x, start=1):
            print(f"x{i} = {nilai:.6f}")

        # Verifikasi otomatis menggunakan fitur bawaan NumPy
        print("\nVerifikasi (numpy.linalg.solve):", np.linalg.solve(A, b))
        print("Residual ||A.x - b|| =", np.linalg.norm(np.array(A) @ x - np.array(b)))

jalankan_aplikasi(): Ini adalah pusat kendali. Di sini program akan bertanya apakah kamu mau memakai contoh soal yang sudah ada, atau mengetik soal baru. Setelah angka didapat, fungsi ini akan "melemparkan" angka tersebut ke fungsi gauss_eliminasi yang sudah kita buat di Sel 2.

Di bagian akhir, program menggunakan np.linalg.solve sebagai kunci jawaban otomatis untuk membuktikan bahwa kode kita menghitung dengan benar.

In [5]:
# Jalankan sel ini untuk memulai program!
jalankan_aplikasi()

=== SPL - Metode Eliminasi Gauss ===
1. Gunakan contoh soal
2. Input sendiri
Pilih (1/2): 1

Contoh soal yang digunakan:
   [2, 1, -1] |  8
   [-3, -1, 2] |  -11
   [-2, 1, 2] |  -3

Matriks augmented awal:
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]] 

Tukar baris R1 <-> R2
R2 <- R2 - (-0.6667) * R1
R3 <- R3 - (0.6667) * R1

Setelah eliminasi kolom 1:
[[ -3.      -1.       2.     -11.    ]
 [  0.       0.3333   0.3333   0.6667]
 [  0.       1.6667   0.6667   4.3333]] 

Tukar baris R2 <-> R3
R3 <- R3 - (0.2000) * R2

Setelah eliminasi kolom 2:
[[ -3.      -1.       2.     -11.    ]
 [  0.       1.6667   0.6667   4.3333]
 [  0.       0.       0.2     -0.2   ]] 

=== HASIL ===
x1 = 2.000000
x2 = 3.000000
x3 = -1.000000

Verifikasi (numpy.linalg.solve): [ 2.  3. -1.]
Residual ||A.x - b|| = 8.881784197001252e-16


Sel ini adalah "tombol on" (saklar). Semua kode yang kita tulis di Sel 1, 2, dan 3 hanya berupa instruksi yang disimpan di memori komputer (belum dijalankan). Dengan memanggil jalankan_aplikasi() di sel ini, kita menyuruh komputer untuk mulai menjalankan semua instruksi tersebut.